In [ ]:
#import pandas
import pandas as pd

admissions = pd.read_csv('../data/processed/admissions_clean.csv')
doctors = pd.read_csv('../data/processed/doctors_preprocessed.csv')
departments = pd.read_csv('../data/processed/departments_clean.csv')

print(admissions.shape)
print(doctors.shape)
print(departments.shape)

(5000, 8)
(500, 5)
(20, 2)


In [2]:
admissions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Admission_ID         5000 non-null   object
 1   Patient_ID           5000 non-null   object
 2   Doctor_ID            5000 non-null   object
 3   Department_ID        5000 non-null   object
 4   Admission_Date       5000 non-null   object
 5   Discharge_Date       5000 non-null   object
 6   Status               5000 non-null   object
 7   Length_of_Stay_Days  5000 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 312.6+ KB


In [3]:
doctors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Doctor_ID       500 non-null    object
 1   Doctor_Name     500 non-null    object
 2   Department_ID   500 non-null    object
 3   Specialization  500 non-null    object
 4   Experience      500 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 19.7+ KB


In [4]:
departments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   department_id    20 non-null     object
 1   department_name  20 non-null     object
dtypes: object(2)
memory usage: 452.0+ bytes


In [5]:
# Merge admissions with doctors on Doctor_ID
df = admissions.merge(doctors, on='Doctor_ID', how='left', suffixes=('', '_doc'))

# Merge with departments (handle lowercase column name)
df = df.merge(departments, left_on='Department_ID', right_on='department_id', how='left')

print(df.shape)
df.head()

(5000, 14)


,Admission_ID,Patient_ID,Doctor_ID,Department_ID,Admission_Date,Discharge_Date,Status,Length_of_Stay_Days,Doctor_Name,Department_ID_doc,Specialization,Experience,department_id,department_name
0,A00001,P00001,DR00186,D002,2025-10-20,2025-10-28,Critical,8,Doctor_186,D002,Neurology,2,D002,Neurology
1,A00002,P00002,DR00281,D019,2025-01-21,2025-01-23,Recovered,2,Doctor_281,D019,Endocrinology,15,D019,Endocrinology
2,A00003,P00003,DR00105,D020,2025-06-24,2025-07-03,Discharged,9,Doctor_105,D020,Dental,33,D020,Dental
3,A00004,P00004,DR00312,D004,2025-07-23,2025-07-25,Critical,2,Doctor_312,D004,Pediatrics,34,D004,Pediatrics
4,A00005,P00005,DR00430,D018,2025-04-06,2025-04-07,Under Treatment,1,Doctor_430,D018,Ophthalmology,4,D018,Ophthalmology


In [6]:
#Analysis 1 : LOS Outliers
Q1 = df['Length_of_Stay_Days'].quantile(0.25)
Q3 = df['Length_of_Stay_Days'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

los_outliers = df[df['Length_of_Stay_Days'] > upper_bound]
print(f"LOS upper bound: {upper_bound}")
print(f"Number of LOS outliers: {len(los_outliers)}")

los_outliers[['Patient_ID', 'department_name', 'Length_of_Stay_Days']].sort_values('Length_of_Stay_Days', ascending=False).head(10)

LOS upper bound: 15.5
Number of LOS outliers: 0


,Patient_ID,department_name,Length_of_Stay_Days


In [7]:
#Analysis 2 : Admissions Per Doctor
admissions_per_doctor = df.groupby('Doctor_Name')['Patient_ID'].count().sort_values(ascending=False)
print("Top 10 busiest doctors:")
admissions_per_doctor.head(10)

Top 10 busiest doctors:


Doctor_Name
Doctor_39     21
Doctor_182    19
Doctor_15     18
Doctor_469    18
Doctor_273    18
Doctor_30     18
Doctor_168    17
Doctor_370    17
Doctor_397    17
Doctor_311    17
Name: Patient_ID, dtype: int64

In [8]:
#Analysis 3 : Admissions per Department over Time
df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])
df['Month'] = df['Admission_Date'].dt.to_period('M')

dept_time_trend = df.groupby(['department_name', 'Month'])['Patient_ID'].count().reset_index()
dept_time_trend = dept_time_trend.rename(columns={'Patient_ID': 'Admission_Count'})
dept_time_trend.head(20)

,department_name,Month,Admission_Count
0,Cardiology,2025-01,19
1,Cardiology,2025-02,19
2,Cardiology,2025-03,29
3,Cardiology,2025-04,20
4,Cardiology,2025-05,28
5,Cardiology,2025-06,18
6,Cardiology,2025-07,22
7,Cardiology,2025-08,31
8,Cardiology,2025-09,21
9,Cardiology,2025-10,26


In [9]:
#Analysis 4 :Critical Status Concentration
print("Unique status values:", df['Status'].unique())  # run this first to confirm exact spelling

Unique status values: ['Critical' 'Recovered' 'Discharged' 'Under Treatment']


In [10]:
critical_df = df[df['Status'] == 'Critical']
critical_by_dept = critical_df.groupby('department_name')['Patient_ID'].count().sort_values(ascending=False)
print("Critical cases by department:")
critical_by_dept

Critical cases by department:


department_name
Orthopedics         92
Emergency           87
Pulmonology         87
Oncology            76
Psychiatry          71
Endocrinology       70
Pediatrics          70
Cardiology          69
Radiology           69
Ophthalmology       69
Dental              69
ICU                 64
Gastroenterology    63
Dermatology         53
ENT                 53
Nephrology          51
Neurology           46
Urology             46
General Surgery     40
Gynecology          34
Name: Patient_ID, dtype: int64

In [11]:
total_by_dept = df.groupby('department_name')['Patient_ID'].count()
critical_pct_by_dept = (critical_by_dept / total_by_dept * 100).sort_values(ascending=False).round(2)
print("Critical case % by department:")
critical_pct_by_dept

Critical case % by department:


department_name
Cardiology          29.61
Dermatology         29.61
Orthopedics         28.75
Psychiatry          28.06
Pulmonology         27.44
Dental              27.27
Urology             27.22
Emergency           25.97
Radiology           25.94
Ophthalmology       25.75
ICU                 25.30
Pediatrics          25.27
ENT                 24.65
Gastroenterology    24.23
Nephrology          23.39
Neurology           23.00
Oncology            22.96
General Surgery     22.86
Endocrinology       21.94
Gynecology          21.38
Name: Patient_ID, dtype: float64

## Findings: Bottlenecks & Capacity Strain

*Data Scope Note:* No wait-time, bed-capacity, or patient-movement/transfer 
fields exist in this dataset. This analysis uses volume-based proxies — LOS 
outliers, admissions-per-doctor, admissions-per-department-over-time, and 
critical-status concentration — as indirect signals of operational strain.

*Key Findings:*

1. *Length of Stay:* No formal statistical outliers were found (IQR upper 
   bound: 15.5 days), indicating stay durations are relatively consistent 
   across patients, with no single department showing extreme prolonged stays.

2. *Doctor Workload:* Doctor_39 handled the highest patient load (21 
   admissions), followed by Doctor_182 (19). Several doctors cluster around 
   17-18 admissions, suggesting workload is fairly evenly distributed with a 
   few doctors carrying slightly more.

3. *Department Trends Over Time:* Admission volumes fluctuate monthly per 
   department (e.g., Cardiology ranged from 18 to 31 admissions/month), 
   indicating seasonal or cyclical demand patterns worth monitoring for 
   staffing decisions.

4. *Critical Case Concentration:* Orthopedics (92) and Emergency (87) have 
   the highest raw counts of critical cases, indicating high absolute strain. 
   However, Cardiology and Dermatology have the highest critical-case 
   percentage (~29.6%), meaning a larger share of their total patients are 
   critical — a stronger relative strain signal despite lower raw volume.

*Conclusion:* Strain is not uniform — it shows up differently depending on 
whether you look at raw volume (Orthopedics, Emergency) or relative severity 
(Cardiology, Dermatology). Both views should be surfaced on the dashboard.

In [13]:
avg_los_by_dept = df.groupby('department_name')['Length_of_Stay_Days'].mean().sort_values(ascending=False).round(2)
print("Average LOS by department:")
avg_los_by_dept

Average LOS by department:


department_name
Nephrology          5.89
Endocrinology       5.67
ICU                 5.66
Psychiatry          5.65
Emergency           5.61
Dermatology         5.59
Radiology           5.57
Urology             5.57
Dental              5.56
Pediatrics          5.54
ENT                 5.52
Orthopedics         5.50
Ophthalmology       5.45
Neurology           5.44
Gynecology          5.43
Oncology            5.40
Cardiology          5.39
General Surgery     5.35
Gastroenterology    5.33
Pulmonology         5.21
Name: Length_of_Stay_Days, dtype: float64

In [15]:
top_los = df[['Patient_ID', 'department_name', 'Length_of_Stay_Days']].sort_values('Length_of_Stay_Days', ascending=False).head(10)
print("Top 10 longest stays:")
top_los

Top 10 longest stays:


,Patient_ID,department_name,Length_of_Stay_Days
4976,P04977,Gastroenterology,10
42,P00043,Ophthalmology,10
1020,P01021,Cardiology,10
1053,P01054,Ophthalmology,10
1012,P01013,Endocrinology,10
4544,P04545,ENT,10
4539,P04540,Emergency,10
4536,P04537,Radiology,10
4964,P04965,Emergency,10
31,P00032,Dental,10


In [16]:
import os
os.makedirs('../outputs', exist_ok=True)

admissions_per_doctor.to_csv('../outputs/member9_admissions_per_doctor.csv')
avg_los_by_dept.to_csv('../outputs/member9_avg_los_by_department.csv')
top_los.to_csv('../outputs/member9_top_los_patients.csv')
critical_by_dept.to_csv('../outputs/member9_critical_by_department.csv')
critical_pct_by_dept.to_csv('../outputs/member9_critical_pct_by_department.csv')
dept_time_trend.to_csv('../outputs/member9_dept_admissions_over_time.csv')

print("All KPI files exported successfully!")

All KPI files exported successfully!
